# 公開重みのモデルで、文章を作る仕組みを観察する

**目的**：トークン分割・入力埋め込み・次の候補を見て、短い質問を試す。LLM説明の補助用（約10分、モデル取得時間は別）。共通論文の初回読解後に使う。

**費用と通信**：モデル取得にはネット接続が必要。推論は選択したMac/Colab上で実行し、OpenAI等のAPIを呼ばない。Colabの利用枠・計算資源は保証されない。

**モデル**：[Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B)、Apache 2.0。0.6Bはモデル名。固定revisionを使う。小型モデルの説明を原論文の正解として使わない。

**保存**：最後にJSONを保存する。計算機が変われば同じseedでも出力が一致するとは限らない。

In [ ]:
from pathlib import Path
import sys, os, json
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src/seminar_lab").exists():
    raise RuntimeError("教材一式を展開したフォルダで実行してください。")
sys.path.insert(0, str(ROOT / "src"))
from seminar_lab.open_weight import configure_cache, MODEL_ID, REVISION
CACHE = configure_cache(ROOT)
print("モデル:", MODEL_ID, "版:", REVISION)
print("キャッシュ:", CACHE)

## 1. 実行環境を明示する

Colabでは「ランタイムのタイプを変更」で **T4 GPU** を選びます（授業で使う機種として指定済み。割当ては画面で確認）。Colab実機検証は未完了です。Macは `mps` を選びます。CPUを試す場合は自分で `cpu` に変更します。失敗時に自動で切り替えません。

依存をまだ入れていない環境では、次の `INSTALL` を `True` に変更します。ローカルでは専用仮想環境を使用してください。

In [ ]:
INSTALL = False
if INSTALL:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--cache-dir", str(CACHE / "pip"),
                    "-r", str(ROOT / "requirements-open-weight.txt")], check=True)
else:
    print("依存のインストールは実行していません。")

In [ ]:
DEVICE = "cuda"  # Colab GPU: cuda / Apple Silicon Mac: mps / CPU: cpu
ALLOW_DOWNLOAD = False  # 最初のモデル取得時だけ明示的にTrueへ
LOAD_MODEL = False
if LOAD_MODEL:
    from seminar_lab.open_weight import load_model
    tokenizer, model = load_model(DEVICE, allow_download=ALLOW_DOWNLOAD)
    print("選択した計算機:", model.device, "数値形式:", model.dtype)
else:
    print("モデルは未読込みです。実行する計算機を選んでからLOAD_MODELをTrueへ。")

## 2. 同じ文章を、文字・ID・数値として見る

2023年講義の身近な例を使います。各トークンは単語と一致するでしょうか。数値の列は入力埋め込みの先頭4要素だけで、文脈を処理した後の表現とは異なります。数値一つに日本語の意味を割り当てないでください。

In [ ]:
OBSERVE = False
TEXT = "京都の名物は"
if OBSERVE:
    if "model" not in globals() or "tokenizer" not in globals():
        raise RuntimeError("モデルを明示的に読み込んでから実行してください。")
    from seminar_lab.open_weight import token_observation, next_token_predictions
    observation = token_observation(tokenizer, model, TEXT)
    print(json.dumps(observation, ensure_ascii=False, indent=2))
    for prefix in [TEXT, "京都の観光名所は"]:
        print(prefix, next_token_predictions(tokenizer, model, prefix))
else:
    print("観察は未実行です。")

## 3. 自分の質問で短い応答を作る

上の候補の確率は「その続きを選ぶ確率」で、事実の正しさを保証する値ではありません。質問を自分で記入し、送信前に会話テンプレートの全文を確認します。ここでは毎回新しい会話です。

思考モードは明示的に無効化します。入力512トークン、出力最大128トークンの短い観察に限定し、超過時は自動で切り詰めません。モデルが出力した文章を、内部の思考過程そのものとは扱いません。

In [ ]:
PROMPT = ""
PREVIEW = False
if PREVIEW:
    if "model" not in globals() or "tokenizer" not in globals():
        raise RuntimeError("モデルを明示的に読み込んでから実行してください。")
    from seminar_lab.open_weight import encode_prompt
    visible_template, encoded = encode_prompt(tokenizer, PROMPT)
    print(visible_template)
    print("入力トークン数:", encoded.input_ids.shape[1])

In [ ]:
RUN = False
if RUN:
    if "model" not in globals() or "tokenizer" not in globals():
        raise RuntimeError("モデルを明示的に読み込んでから実行してください。")
    from seminar_lab.open_weight import generate
    result = generate(tokenizer, model, PROMPT, max_new_tokens=96, seed=42)
    print(result["response_raw"])
    print("終了状態:", result["status"], "秒:", result["elapsed_seconds"])
    if result["status"] == "output_limit":
        print("出力上限で終了。完了した回答として扱わず、必要なら次に条件を見直します。")
else:
    print("生成は未実行です。")

## 4. 生回答と自分の理解を分けて保存する

共通論文の朝・講義後の記録へ上書きせず、仕組みの観察として別に保存します。Colabでは保存したJSONをFiles欄からダウンロードしてください。

In [ ]:
SAVE = False
if SAVE:
    if "result" not in globals():
        raise RuntimeError("保存する生成結果がありません。")
    result["student_explanation"] = ""  # 自分が説明できたこと
    result["unresolved_point"] = ""  # 確認できていないこと
    folder = ROOT / "outputs/open-weight"
    if not folder.resolve().is_relative_to(ROOT):
        raise ValueError("保存先は教材フォルダ内にしてください。")
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / (result["run_id"] + ".json")
    with destination.open("x", encoding="utf-8") as handle:
        json.dump(result, handle, ensure_ascii=False, indent=2)
    print("保存:", destination)